In [ ]:
import numpy as np 
import pandas as pd 
import json
import os

#iopub.status.idle":"2025-10-28T09:00:35.258756Z","shell.execute_reply.started":"2025-10-28T09:00:31.445478Z","shell.execute_reply":"2025-10-28T09:00:35.258140Z"}}
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset


#For debugging purposes
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load and preprocess the data
corpus = pd.read_csv("encrypted_corpus.csv")
all_ciphers = set("".join(corpus["ciphers"].astype(str).tolist()))
all_plain = set("".join(corpus["word"].astype(str).tolist()))

cipher_to_idx = {char: idx for idx, char in enumerate(sorted(list(all_ciphers)))}
plain_to_idx = {char: idx for idx, char in enumerate(sorted(list(all_plain)))}

# Add special tokens
for tok in ("<PAD>", "<SOS>", "<EOS>"):
    if tok not in plain_to_idx:
        plain_to_idx[tok] = len(plain_to_idx)

pad_idx = plain_to_idx["<PAD>"]
sos_idx = plain_to_idx["<SOS>"]
eos_idx = plain_to_idx["<EOS>"]

cipher_lookup_size = len(cipher_to_idx)
plain_lookup_size = len(plain_to_idx)
embedding_dim = 64
hidden_size = 128

# Model components
cipher_embedding = nn.Embedding(cipher_lookup_size, embedding_dim)
plain_embedding = nn.Embedding(plain_lookup_size, embedding_dim)
encoder_lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
decoder_lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
output_proj = nn.Linear(hidden_size, plain_lookup_size)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
params = (
    list(cipher_embedding.parameters())
    + list(plain_embedding.parameters())
    + list(encoder_lstm.parameters())
    + list(decoder_lstm.parameters())
    + list(output_proj.parameters())
)
optimizer = optim.Adam(params, lr=1e-3)

#move these tensors to a gpu
cipher_embedding = cipher_embedding.to(device)
plain_embedding = plain_embedding.to(device)
encoder_lstm = encoder_lstm.to(device)
decoder_lstm = decoder_lstm.to(device)
output_proj = output_proj.to(device)


/kaggle/input/corpus/encrypted_corpus.csv
Using device: cuda


In [ ]:
import wandb

# Set your WandB API key as an environment variable before running:
wandb.login(key=os.environ.get("WANDB_API_KEY"))
wandb.init(project="Stryder", entity="k-kasozi69-horizon-international-school-uganda", reinit=True)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: k-kasozi69 (k-kasozi69-horizon-international-school-uganda) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run 7nvzzir1
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260329_134501-7nvzzir1
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run generous-armadillo-6
wandb: ⭐️ View project at https://wandb.ai/k-kasozi69-horizon-international-school-uganda/Stryder
wandb: 🚀 View run at https://wandb.ai/k-kasozi69-horizon-international-school-uganda/Stryder/runs/7nvzzir1


In [ ]:
# Dataset class with <SOS>/<EOS> handling
class CipherDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.samples = []
        for _, row in df.iterrows():
            #encoding as per the dictionaries
            cipher_seq = [cipher_to_idx[ch] for ch in str(row["ciphers"]) if ch in cipher_to_idx]
            plain_seq = [plain_to_idx[ch] for ch in str(row["word"]) if ch in plain_to_idx]
            if len(cipher_seq) == 0 or len(plain_seq) == 0:
                continue

            cipher_tensor = torch.tensor(cipher_seq, dtype=torch.long)
            plain_tensor = torch.tensor(plain_seq, dtype=torch.long)

            # Add <SOS> for decoder input and <EOS> for target
            decoder_input = torch.cat((torch.tensor([sos_idx]), plain_tensor))
            target = torch.cat((plain_tensor, torch.tensor([eos_idx])))

            self.samples.append((cipher_tensor, decoder_input, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# Collate function for three sequences
def collate_fn(batch):
    ciphers, dec_ins, targets = zip(*batch)
    cipher_padded = pad_sequence(ciphers, batch_first=True, padding_value=pad_idx)
    dec_in_padded = pad_sequence(dec_ins, batch_first=True, padding_value=pad_idx)
    target_padded = pad_sequence(targets, batch_first=True, padding_value=pad_idx)
    return cipher_padded, dec_in_padded, target_padded

dataset = CipherDataset(corpus)
loader = DataLoader(dataset, batch_size=256, shuffle=True, collate_fn=collate_fn)
global_step = 0

for epoch in range(100):
    total_loss = 0.0

    for cipher_batch, dec_in_batch, target_batch in loader:
        cipher_batch = cipher_batch.to(device)
        dec_in_batch = dec_in_batch.to(device)
        target_batch = target_batch.to(device)

        cipher_embed = cipher_embedding(cipher_batch)
        _, (hn, cn) = encoder_lstm(cipher_embed)

        decoder_embed = plain_embedding(dec_in_batch)
        decoder_outputs, _ = decoder_lstm(decoder_embed, (hn, cn))

        logits = output_proj(decoder_outputs)
        loss = criterion(logits.view(-1, plain_lookup_size), target_batch.view(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Log batch-level loss on its own step
        wandb.log({"train/batch_loss": loss.item()}, step=global_step)
        global_step += 1

    avg_loss = total_loss / len(loader)

    # Epoch-level metrics get logged separately, keyed by epoch not global_step
    wandb.log({"train/epoch_loss": avg_loss,"epoch": epoch + 1})
    print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}")


save_path = "/kaggle/working/cipher_model.pt"

torch.save({
    "cipher_embedding": cipher_embedding.state_dict(),
    "plain_embedding": plain_embedding.state_dict(),
    "encoder_lstm": encoder_lstm.state_dict(),
    "decoder_lstm": decoder_lstm.state_dict(),
    "output_proj": output_proj.state_dict(),
    "optimizer": optimizer.state_dict()
}, save_path)

print(f"Model saved to {save_path}")

wandb.finish()


Epoch 1, Avg Loss: 1.7642
Epoch 2, Avg Loss: 0.6842
Epoch 3, Avg Loss: 0.4055
Epoch 4, Avg Loss: 0.2972
Epoch 5, Avg Loss: 0.2398
Epoch 6, Avg Loss: 0.2043
Epoch 7, Avg Loss: 0.1781
Epoch 8, Avg Loss: 0.1579
Epoch 9, Avg Loss: 0.1393
Epoch 10, Avg Loss: 0.1228
Epoch 11, Avg Loss: 0.1079
Epoch 12, Avg Loss: 0.0953
Epoch 13, Avg Loss: 0.0837
Epoch 14, Avg Loss: 0.0735
Epoch 15, Avg Loss: 0.0649
Epoch 16, Avg Loss: 0.0573
Epoch 17, Avg Loss: 0.0508
Epoch 18, Avg Loss: 0.0455
Epoch 19, Avg Loss: 0.0411
Epoch 20, Avg Loss: 0.0376
Epoch 21, Avg Loss: 0.0347
Epoch 22, Avg Loss: 0.0317
Epoch 23, Avg Loss: 0.0299
Epoch 24, Avg Loss: 0.0278
Epoch 25, Avg Loss: 0.0263
Epoch 26, Avg Loss: 0.0245
Epoch 27, Avg Loss: 0.0233
Epoch 28, Avg Loss: 0.0217
Epoch 29, Avg Loss: 0.0204
Epoch 30, Avg Loss: 0.0197
Epoch 31, Avg Loss: 0.0186
Epoch 32, Avg Loss: 0.0175
Epoch 33, Avg Loss: 0.0169
Epoch 34, Avg Loss: 0.0163
Epoch 35, Avg Loss: 0.0154
Epoch 36, Avg Loss: 0.0148
Epoch 37, Avg Loss: 0.0145
Epoch 38, 

wandb: updating run metadata


Epoch 100, Avg Loss: 0.0039
Model saved to /kaggle/working/cipher_model.pt


wandb: 
wandb: Run history:
wandb:            epoch ▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇██
wandb: train/batch_loss █▄▄▂▂▂▂▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: train/epoch_loss █▇▅▅▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:            epoch 100
wandb: train/batch_loss 0.00466
wandb: train/epoch_loss 0.00388
wandb: 
wandb: 🚀 View run generous-armadillo-6 at: https://wandb.ai/k-kasozi69-horizon-international-school-uganda/Stryder/runs/7nvzzir1
wandb: ⭐️ View project at: https://wandb.ai/k-kasozi69-horizon-international-school-uganda/Stryder
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260329_134501-7nvzzir1/logs
